# 📊 NeMo Fraud Detection: MLflow Evaluierung (Zero-Shot & Few-Shot)

Dieses Notebook evaluiert das Basismodell (**Llama-3.1-8B-Instruct** via NVIDIA NIM) in den Modi **Zero-Shot** und **Few-Shot** zur Betrugserkennung.

### Funktionsumfang:
1. **Modus-Steuerung:** Testet das Modell wahlweise mit reiner Aufgabenstellung (Zero-Shot) oder inkl. Positiv-/Negativ-Beispielen (Few-Shot).
2. **API-Kommunikation:** Fragt den lokalen NIM-Endpunkt per Chat-Completion ab.
3. **Metrik-Berechnung & Tracking:** Nutzt `scikit-learn` für detaillierte Metriken (Accuracy, Precision, Recall, F1-Score) und protokolliert die Ergebnisse automatisch in **MLflow**.

In [1]:
import json
import re
import requests
import mlflow
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support

# Konfiguration
VAL_FILE = "/data/nemo-fraud-detection/notebooks/02_Data_Curation/data/sft/validation.jsonl"
NIM_URL = "http://172.17.0.1:8800/v1/chat/completions"

# MLflow Experiment konfigurieren
mlflow.set_experiment("NeMo-Fraud-Detection-Evaluation")
print("✅ Bibliotheken geladen und MLflow-Experiment gesetzt.")

/usr/local/lib/python3.10/dist-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/08/22 14:14:30 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/22 14:14:30 INFO mlflow.store.db.utils: Updating database tables
2026/08/22 14:14:32 INFO mlflow.tracking.fluent: Experiment with name 'NeMo-Fraud-Detection-Evaluation' does not exist. Creating a new experiment.


✅ Bibliotheken geladen und MLflow-Experiment gesetzt.


### 1. Evaluierungs- und Logging-Funktion
Diese Funktion steuert den Durchlauf für den jeweiligen Modus (`zero-shot` oder `few-shot`), führt das Modell aus, parst die Antworten und loggt alle Kennzahlen direkt in MLflow.

In [2]:
def evaluate_and_log(mode="few-shot"):
    print(f"\n🚀 Starte Evaluierung im Modus: {mode.upper()} mit MLflow Tracking...\n")
    
    y_true = []
    y_pred = []
    total = 0
    passed_judge = 0
    
    with open(VAL_FILE, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            transcript_text = data.get("input", "")
            true_label = data.get("output", "").strip().lower()
            
            if not transcript_text:
                continue
            
            total += 1
            
            # Messages je nach Modus aufbauen
            messages = [
                {
                    "role": "system",
                    "content": "Du bist ein Experte für Betrugserkennung. Analysiere das Transkript kurz, begründe deine Entscheidung und beende deine Antwort mit exakt 'Result: fraud' oder 'Result: legitimate'."
                }
            ]
            
            if mode == "few-shot":
                messages.extend([
                    {
                        "role": "user",
                        "content": "Kunde: Hallo, ich wollte nur fragen, wann meine nächste Rechnung abgebucht wird. Agent: Guten Tag, die Abbuchung erfolgt immer am Monatsletzten."
                    },
                    {
                        "role": "assistant",
                        "content": "Analyse: Normale Anfrage zur Rechnung ohne Betrugsindizien. Result: legitimate"
                    },
                    {
                        "role": "user",
                        "content": "Kunde: Mein Konto ist komplett leergeräumt, geben Sie sofort die TAN frei!"
                    },
                    {
                        "role": "assistant",
                        "content": "Analyse: Panikmache und Druck auf den Agenten zur TAN-Freigabe. Klassischer Betrug. Result: fraud"
                    }
                ])
            
            messages.append({"role": "user", "content": transcript_text})
            
            payload = {
                "model": "meta/llama-3.1-8b-instruct",
                "messages": messages,
                "temperature": 0.0,
                "max_tokens": 150
            }
            
            try:
                response = requests.post(NIM_URL, json=payload)
                result = response.json()
                model_output = result["choices"][0]["message"]["content"].strip().lower()
                
                # Parsing des Ergebnisses
                if "result: legitimate" in model_output and "fraud" not in model_output:
                    model_answer = "legitimate"
                elif "result: fraud" in model_output and "legitimate" not in model_output:
                    model_answer = "fraud"
                else:
                    match = re.search(r'\b(fraud|legitimate)\b', model_output)
                    model_answer = match.group(1) if match else "unknown"
                
                y_true.append(true_label)
                y_pred.append(model_answer)
                
                if model_answer == true_label:
                    passed_judge += 1
                    
            except Exception as e:
                print(f"Fehler bei Eintrag {total}: {e}")

    # Metriken berechnen
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=["fraud", "legitimate"], average="weighted", zero_division=0
    )
    judge_rate = (passed_judge / total) * 100 if total > 0 else 0

    print(f"📊 ERGEBNISSE FÜR {mode.upper()}:")
    print(f"Accuracy: {acc * 100:.2f}% | F1-Score: {f1:.4f} | Match-Rate: {judge_rate:.2f}%")
    print("-" * 40)
    print(classification_report(y_true, y_pred, labels=["fraud", "legitimate"], target_names=["fraud", "legitimate"], zero_division=0))

    # In MLflow einloggen
    with mlflow.start_run(run_name=f"evaluation-{mode}"):
        mlflow.log_param("evaluation_mode", mode)
        mlflow.log_param(
            "model_name", "meta/llama-3.1-8b-instruct"
        )
        mlflow.log_param("dataset_file", VAL_FILE)
        
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("weighted_f1", f1)
        mlflow.log_metric("weighted_precision", precision)
        mlflow.log_metric("weighted_recall", recall)
        mlflow.log_metric("match_rate_percentage", judge_rate)
        
        print(f"✅ Ergebnisse erfolgreich an MLflow übergeben für Run-Modus: {mode}\n")

### 2. Ausführung der Experimente (Zero-Shot & Few-Shot)
Hier werden nacheinander beide Modi ausgeführt, damit du die Resultate im MLflow-UI direkt miteinander vergleichen kannst.

In [3]:
# Beide Modi nacheinander tracken
evaluate_and_log(mode="zero-shot")
evaluate_and_log(mode="few-shot")


🚀 Starte Evaluierung im Modus: ZERO-SHOT mit MLflow Tracking...

📊 ERGEBNISSE FÜR ZERO-SHOT:
Accuracy: 0.00% | F1-Score: 0.0000 | Match-Rate: 0.00%
----------------------------------------
              precision    recall  f1-score   support

       fraud       0.00      0.00      0.00      12.0
  legitimate       0.00      0.00      0.00      10.0

   micro avg       0.00      0.00      0.00      22.0
   macro avg       0.00      0.00      0.00      22.0
weighted avg       0.00      0.00      0.00      22.0

✅ Ergebnisse erfolgreich an MLflow übergeben für Run-Modus: zero-shot


🚀 Starte Evaluierung im Modus: FEW-SHOT mit MLflow Tracking...

📊 ERGEBNISSE FÜR FEW-SHOT:
Accuracy: 45.45% | F1-Score: 0.6239 | Match-Rate: 45.45%
----------------------------------------
              precision    recall  f1-score   support

       fraud       1.00      0.42      0.59        12
  legitimate       1.00      0.50      0.67        10

   micro avg       1.00      0.45      0.62        22
   m